# Notebook 07 - Load Pre-trained Model (Transfer Learning)

Import a pre-trained model (ResNet50 or MobileNet) with ImageNet weights, freeze all backbone/base layers, and prepare the model for fine-tuning on the PlantVillage crop disease dataset.

**Topic Covered:**
- Pre-trained model loaded with ImageNet weights
- All backbone layers frozen
- New classification head attached
- Model moved to GPU/CPU
- Forward pass sanity check
- Config saved for next steps

## Import Packages

In [1]:
import torch
import torch.nn as nn
import torchvision
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader

from pathlib import Path
import json
import sys

## Dataset Configuration

In [2]:
# ─── CONFIG ───
from pathlib import Path
import os, torch
from torchvision import transforms

def get_project_root():
    if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
        nb_dir = Path(os.getcwd())
    else:
        import IPython
        ip = IPython.get_ipython()
        nb_dir = Path(ip.run_line_magic('pwd', '')).resolve() if ip else Path('.').resolve()
    return nb_dir.parent

PROJECT_ROOT = get_project_root()
DATA_DIR     = PROJECT_ROOT / 'data'
MODELS_DIR   = PROJECT_ROOT / 'models'
SPLITS_DIR   = DATA_DIR / 'splits'
RESULTS_DIR  = PROJECT_ROOT / 'results'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_WORKERS  = 2
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'SPLITS_DIR   : {SPLITS_DIR}')
print(f'Device       : {DEVICE}')
print('Transforms defined successfully ✅')


PROJECT_ROOT : /content/drive/MyDrive/Project2_AgroLens_AI
SPLITS_DIR   : /content/drive/MyDrive/Project2_AgroLens_AI/data/splits
Device       : cuda
Transforms defined successfully ✅


## Load Dataset

In [3]:
import sys
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# ─── Define PlantVillageDataset inline (no utils.py needed) ──────────────────
class PlantVillageDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df        = pd.read_csv(csv_path)
        self.transform = transform
        self.classes   = sorted(self.df['label'].unique().tolist())
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['filepath']).convert('RGB')
        label = self.class_to_idx[row['label']]
        if self.transform:
            image = self.transform(image)
        return image, label

# ─── Load datasets from CSV ───────────────────────────────────────────────────
try:
    train_dataset = PlantVillageDataset(SPLITS_DIR / 'train.csv', transform=train_transforms)
    val_dataset   = PlantVillageDataset(SPLITS_DIR / 'val.csv',   transform=val_transforms)
    test_dataset  = PlantVillageDataset(SPLITS_DIR / 'test.csv',  transform=val_transforms)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    NUM_CLASSES = len(train_dataset.classes)

    print(f'\n📊 Dataset Summary:')
    print(f'Training samples   : {len(train_dataset)}')
    print(f'Validation samples : {len(val_dataset)}')
    print(f'Test samples       : {len(test_dataset)}')
    print(f'Number of classes  : {NUM_CLASSES}')
    print(f'Classes            : {train_dataset.classes}')

except Exception as e:
    print(f"❌ Error loading datasets: {e}")
    raise



📊 Dataset Summary:
Training samples   : 14446
Validation samples : 3096
Test samples       : 3096
Number of classes  : 15
Classes            : ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


## Load Pre-trained Model with ImageNet Weights

Choose between **ResNet50** and **MobileNetV2**. ResNet50 is more accurate; MobileNetV2 is lighter and faster — useful for mobile deployment.

In [4]:
# ─── Choose your model ────────────────────────────────
MODEL_CHOICE = 'resnet50'   # Options: 'resnet50' or 'mobilenet'
print(f'Model architecture selected: {MODEL_CHOICE.upper()}')

if MODEL_CHOICE == 'resnet50':
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    print('Loaded ResNet50 with ImageNet weights ✅')
elif MODEL_CHOICE == 'mobilenet':
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    print('Loaded MobileNetV2 with ImageNet weights ✅')
else:
    raise ValueError(f'Unknown model choice: {MODEL_CHOICE}')

Model architecture selected: RESNET50


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


  0%|          | 0.00/97.8M [00:00<?, ?B/s]

  1%|          | 768k/97.8M [00:00<00:13, 7.68MB/s]

 12%|█▏        | 11.4M/97.8M [00:00<00:01, 68.0MB/s]

...

 61%|██████    | 59.6M/97.8M [00:00<00:00, 147MB/s]

 77%|███████▋  | 75.5M/97.8M [00:00<00:00, 153MB/s]

 94%|█████████▎| 91.6M/97.8M [00:00<00:00, 158MB/s]

100%|██████████| 97.8M/97.8M [00:00<00:00, 138MB/s]

Loaded ResNet50 with ImageNet weights ✅


## Freeze All Base (Backbone) Layers

By setting `requires_grad = False` on all parameters, we prevent the pre-trained weights from being updated during the initial training phase. Only the new classification head we attach will be trainable.

In [5]:
# Freeze all backbone parameters
for param in model.parameters():
    param.requires_grad = False

# Count frozen vs trainable params before replacing head
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'All backbone layers frozen ❄️')
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

All backbone layers frozen ❄️
Total parameters    : 25,557,032
Trainable parameters: 0


## Replace Classification Head with Crop Disease Head

The final fully-connected layer of the pre-trained model outputs 1000 classes (ImageNet). We replace it with a new head that outputs `NUM_CLASSES` — the number of crop disease categories in our dataset.

In [6]:
if MODEL_CHOICE == 'resnet50':
    # ResNet50: replace the final `fc` layer
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, NUM_CLASSES)
    )
    print(f'ResNet50 FC head replaced → output: {NUM_CLASSES} classes ✅')

elif MODEL_CHOICE == 'mobilenet':
    # MobileNetV2: replace the final classifier layer
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, NUM_CLASSES)
    )
    print(f'MobileNetV2 classifier head replaced → output: {NUM_CLASSES} classes ✅')

# Move model to GPU/CPU
model = model.to(DEVICE)

# Count trainable parameters after head replacement
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params     = sum(p.numel() for p in model.parameters())

print(f'\nAfter head replacement:')
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}  (new head only)')
print(f'Frozen parameters   : {total_params - trainable_params:,}')

ResNet50 FC head replaced → output: 15 classes ✅



After head replacement:
Total parameters    : 24,036,431
Trainable parameters: 528,399  (new head only)
Frozen parameters   : 23,508,032


## Model Summary — Verify Architecture

In [7]:
# Print a layer-level summary showing frozen vs trainable layers
print(f"{'Layer Name':<40} {'Output Shape':<30} {'Trainable':<10}")
print('-' * 80)

for name, param in model.named_parameters():
    status = '✅ Yes' if param.requires_grad else '❄️  Frozen'
    print(f"{name:<40} {str(list(param.shape)):<30} {status}")

Layer Name                               Output Shape                   Trainable 
--------------------------------------------------------------------------------
conv1.weight                             [64, 3, 7, 7]                  ❄️  Frozen
bn1.weight                               [64]                           ❄️  Frozen
bn1.bias                                 [64]                           ❄️  Frozen
layer1.0.conv1.weight                    [64, 64, 1, 1]                 ❄️  Frozen
layer1.0.bn1.weight                      [64]                           ❄️  Frozen
layer1.0.bn1.bias                        [64]                           ❄️  Frozen
layer1.0.conv2.weight                    [64, 64, 3, 3]                 ❄️  Frozen
layer1.0.bn2.weight                      [64]                           ❄️  Frozen
layer1.0.bn2.bias                        [64]                           ❄️  Frozen
layer1.0.conv3.weight                    [256, 64, 1, 1]                ❄️  Frozen
layer1

## Sanity Check — Forward Pass Test

Pass a dummy batch through the model to confirm the output shape matches `NUM_CLASSES`.

In [8]:
model.eval()
with torch.no_grad():
    dummy_input  = torch.randn(4, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    dummy_output = model(dummy_input)

print(f'Input shape  : {list(dummy_input.shape)}')
print(f'Output shape : {list(dummy_output.shape)}')
assert dummy_output.shape == (4, NUM_CLASSES), 'Output shape mismatch!'
print(f'\nSanity check passed ✅ — Model outputs {NUM_CLASSES} class logits')

Input shape  : [4, 3, 224, 224]
Output shape : [4, 15]

Sanity check passed ✅ — Model outputs 15 class logits


## Save Model Config for Next Steps

In [9]:
config = {
    'model_choice'  : MODEL_CHOICE,
    'num_classes'   : NUM_CLASSES,
    'class_names'   : train_dataset.classes,
    'img_size'      : IMG_SIZE,
    'batch_size'    : BATCH_SIZE,
    'frozen_backbone': True,
    'imagenet_mean' : IMAGENET_MEAN,
    'imagenet_std'  : IMAGENET_STD
}

with open(MODELS_DIR / 'model_config.json', 'w') as f:
    json.dump(config, f, indent=4)

print('Model config saved to model_config.json ✅')

Model config saved to model_config.json ✅
